# SentinelML GenAI Explanations

Convert tuned XGBoost SHAP case studies into plain-English analyst rationales using Groq.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import shap

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.explain import explain_single_prediction, generate_explanation, validate_explanation_accuracy
from src.train_models import _build_xgboost, _compute_scale_pos_weight, load_processed_splits

## Train Tuned XGBoost

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_processed_splits(
    PROJECT_ROOT / "data" / "processed"
)

best_params = {
    "max_depth": 6,
    "learning_rate": 0.04087659627832751,
    "n_estimators": 600,
    "subsample": 0.8654555403181324,
    "colsample_bytree": 0.9958149399088472,
    "min_child_weight": 8,
}
model = _build_xgboost(_compute_scale_pos_weight(y_train), **best_params)
model.fit(X_train, y_train)
y_val_proba = model.predict_proba(X_val)[:, 1]
y_val_pred = y_val_proba >= 0.5
explainer = shap.TreeExplainer(model)

## Case Study Explanations

In [ ]:
case_indices = {
    "True positive fraud catch": X_val.index[(y_val == 1) & y_val_pred][0],
    "False positive normal flagged as fraud": X_val.index[(y_val == 0) & y_val_pred][0],
    "False negative missed fraud": X_val.index[(y_val == 1) & ~y_val_pred][0],
}

for label, idx in case_indices.items():
    row = X_val.loc[[idx]]
    position = X_val.index.get_loc(idx)
    prediction_prob = float(y_val_proba[position])
    single = explain_single_prediction(model, explainer, row, X_val.columns)
    explanation_text = generate_explanation(single["breakdown"], prediction_prob, row.iloc[0])
    validate_explanation_accuracy(explanation_text, single["breakdown"])

    print(f"{label}: index={idx}, actual={int(y_val.loc[idx])}, p_fraud={prediction_prob:.6f}")
    print(explanation_text)
    display(single["breakdown"].head(8))

    row_shap = explainer.shap_values(row)
    waterfall = shap.Explanation(
        values=row_shap[0],
        base_values=explainer.expected_value,
        data=row.iloc[0],
        feature_names=X_val.columns,
    )
    shap.plots.waterfall(waterfall, max_display=12, show=True)